In [ ]:
#include <TinyGPS++.h>
#include <math.h>

// ================= GPS =================
TinyGPSPlus gps;

// ================= GSR =================
const int GSRPIN = A0;

// ================= Index =================
unsigned long indexCount = 0;

// ================= SCR / Slope / Recovery =================
float gsr_baseline = NAN;
float scr = 0.0f;
float prev_scr = 0.0f;
unsigned long prev_ms = 0;
float slope_cps = 0.0f;

bool in_event = false;
unsigned long recovery_start_ms = 0;
float recovery_time_s = -1.0f;

const float BASELINE_ALPHA = 0.01f;
const float EVENT_ON_TH = 8.0f;
const float EVENT_OFF_TH = 3.0f;
const float RECOVER_BAND = 2.0f;

void setup() {
  Serial.begin(9600);    // 串口监视器
  Serial1.begin(9600);   // GPS 模块接 Serial1

  while (!Serial) {}

  Serial.println("GSR + GPS System Start");
  Serial.println("index,gpsTimeValid,date,time,millis,lat,lng,speed_mph,gsr,scr,slope_cps,recovery_s");

  prev_ms = millis();
}

void loop() {
  // ---------- Read GPS stream ----------
  while (Serial1.available()) {
    gps.encode(Serial1.read());
  }

  // ---------- Read GSR (50-sample average) ----------
  long gsr_sum = 0;
  for (int i = 0; i < 50; i++) {
    gsr_sum += analogRead(GSRPIN);
    delay(5);
  }
  int gsr_average = gsr_sum / 50;

  // ---------- SCR / baseline / slope / recovery ----------
  unsigned long now_ms = millis();
  float dt_s = (now_ms - prev_ms) / 1000.0f;
  if (dt_s <= 0) dt_s = 0.001f;

  if (isnan(gsr_baseline)) {
    gsr_baseline = (float)gsr_average;
  } else {
    gsr_baseline = (1.0f - BASELINE_ALPHA) * gsr_baseline + BASELINE_ALPHA * (float)gsr_average;
  }

  scr = (float)gsr_average - gsr_baseline;
  slope_cps = (scr - prev_scr) / dt_s;

  if (!in_event && scr >= EVENT_ON_TH) {
    in_event = true;
    recovery_time_s = -1.0f;
  }

  if (in_event && scr <= EVENT_OFF_TH) {
    in_event = false;
    recovery_start_ms = now_ms;
    recovery_time_s = 0.0f;
  }

  if (recovery_time_s >= 0.0f) {
    recovery_time_s = (now_ms - recovery_start_ms) / 1000.0f;
    if (fabs(scr) <= RECOVER_BAND) {
      recovery_start_ms = now_ms;
    }
  }

  prev_scr = scr;
  prev_ms = now_ms;

  // ---------- Check GPS time ----------
  bool gpsTimeValid = gps.time.isValid() && gps.date.isValid();

  // ---------- Serial output ----------
  String line = "";
  line += String(indexCount); line += ",";
  line += (gpsTimeValid ? "1," : "0,");

  if (gpsTimeValid) {
    line += String(gps.date.value()); line += ",";
    line += String(gps.time.value()); line += ",";
  } else {
    line += "0,0,";
  }

  line += String(now_ms); line += ",";

  line += (gps.location.isValid() ? String(gps.location.lat(), 6) : "0"); line += ",";
  line += (gps.location.isValid() ? String(gps.location.lng(), 6) : "0"); line += ",";
  line += (gps.speed.isValid() ? String(gps.speed.mph()) : "0"); line += ",";

  line += String(gsr_average); line += ",";
  line += String(scr, 2); line += ",";
  line += String(slope_cps, 2); line += ",";
  line += String(recovery_time_s, 2);

  Serial.println(line);

  indexCount++;
}